# Build a SQL AI agent from scratch

In this notebook, we will build the smallest complete SQL AI agent:

`question → prompt → model-generated SQL → DuckDB → result`

We will use the OpenAI Python SDK to send a prompt to a model and Ibis to execute the returned SQL against the persistent DuckDB database created in the previous tutorial.

## Prerequisites

This notebook expects the environment and database from [`03a_duckdb_settings.ipynb`](03a_duckdb_settings.ipynb). Before starting Jupyter, set the database directory and OpenAI API key in the shell:

```bash
export DUCKDB_DATABASES_PATH="$HOME/databases/duckdb"
export OPENAI_API_KEY="your_api_key_here"
```

The OpenAI SDK reads `OPENAI_API_KEY` from the environment automatically. The dependencies used here are already declared in [`docker/requirements.txt`](../docker/requirements.txt).

## Import the dependencies

We need `os` and `pathlib` for the database path, `re` for the small SQL extraction helper, Ibis for DuckDB, and the official OpenAI SDK for the model request.

In [10]:
import os
import re

from pathlib import Path

import ibis
from openai import OpenAI

## Connect to the existing DuckDB database

The previous tutorial stored the `air_traffic` table in `sql-ai-agent.duckdb`. We reconnect to that file instead of loading and preparing the source CSV again.

In [11]:
database_directory = os.environ.get("DUCKDB_DATABASES_PATH")
if not database_directory:
    raise EnvironmentError(
        "Set DUCKDB_DATABASES_PATH to the directory used in the previous tutorial."
    )

database_path = Path(database_directory).expanduser() / "sql-ai-agent.duckdb"
if not database_path.is_file():
    raise FileNotFoundError(
        f"Could not find {database_path}. Run 03a_duckdb_settings.ipynb first."
    )

connection = ibis.duckdb.connect(database_path)
table_name = "air_traffic"

if table_name not in connection.list_tables():
    raise RuntimeError(
        f"Could not find the {table_name!r} table. Run the previous tutorial first."
    )

print(f"Connected to {database_path.name}")
print(f"Available table: {table_name}")

Connected to sql-ai-agent.duckdb
Available table: air_traffic


## Extract the table schema

A model cannot inspect our database on its own. It needs the table name, column names, and DuckDB data types as prompt context. We retrieve those details from DuckDB's `information_schema.columns` view and format them as a compact schema description.

In [12]:
schema_sql = f"""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = '{table_name}'
ORDER BY ordinal_position
"""

schema_details = connection.sql(schema_sql).execute()
schema_context = ",\n".join(
    f'    "{row.column_name}" {row.data_type}'
    for row in schema_details.itertuples(index=False)
)

print(schema_context)

    "Year" BIGINT,
    "Date" TIMESTAMP,
    "Operating Airline" VARCHAR,
    "Operating Airline IATA Code" VARCHAR,
    "Published Airline" VARCHAR,
    "Published Airline IATA Code" VARCHAR,
    "GEO Summary" VARCHAR,
    "GEO Region" VARCHAR,
    "Activity Type Code" VARCHAR,
    "Price Category Code" VARCHAR,
    "Terminal" VARCHAR,
    "Boarding Area" VARCHAR,
    "Passenger Count" BIGINT


## Set up the OpenAI client

We use `os.getenv()` to load `OPENAI_API_KEY` from the environment and pass it to `OpenAI()` when we create the client. We will use the [Responses API](https://developers.openai.com/api/docs/guides/text) for this direct text-generation request.

In [13]:
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise EnvironmentError("Set OPENAI_API_KEY before running the notebook.")

client = OpenAI(api_key=openai_api_key)
model = "gpt-5.4-mini"

## Build the prompts

The system prompt gives the model its task, the DuckDB dialect, the available table, the schema, and the required output format. The user prompt contains the natural-language question we want to answer.

In [14]:
system_prompt = f"""
Translate the user's question into one DuckDB SQL query.
Return only the SQL query as plain text, without Markdown or an explanation.
Use only the table and columns shown below.
Enclose table and column names in double quotes.
Write a SELECT query that reads data; do not modify the database.

CREATE TABLE "{table_name}" (
{schema_context}
);
""".strip()

question = "What was the total passenger count in 2024?"
user_prompt = f"Write a SQL query that answers: {question}"

## Send the prompt to the model

The Responses API accepts the higher-priority system instructions separately from the user input. The SDK's `output_text` helper returns the generated text from the response.

In [15]:
response = client.responses.create(
    model=model,
    instructions=system_prompt,
    input=user_prompt,
)

model_output = response.output_text
print(model_output)

SELECT SUM("Passenger Count") AS "total passenger count"
FROM "air_traffic"
WHERE "Year" = 2024;


## Extract the SQL

The prompt asks for plain SQL, but a model may still wrap its answer in a Markdown code fence. This small helper removes that optional wrapper and surrounding whitespace. It is a parser, not a security validator.

In [16]:
def extract_sql(text):
    fenced_sql = re.search(
        r"```(?:sql)?\s*(.*?)```",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    sql = fenced_sql.group(1) if fenced_sql else text
    sql = sql.strip()

    if not sql:
        raise ValueError("The model did not return a SQL query.")

    return sql


generated_sql = extract_sql(model_output)
print(generated_sql)

SELECT SUM("Passenger Count") AS "total passenger count"
FROM "air_traffic"
WHERE "Year" = 2024;


## Send the SQL to DuckDB

Finally, we pass the generated SQL to the same Ibis connection and ask DuckDB to execute it.

In [17]:
result = connection.sql(generated_sql).execute()
result

,total passenger count
0,52210939


## Put the workflow into one function

The individual cells show every step. We can now wrap the same sequence in a small function so another question can travel through the complete agent workflow.

In [20]:
def ask_database(question):
    response = client.responses.create(
        model=model,
        instructions=system_prompt,
        input=f"Write a SQL query that answers: {question}",
    )
    sql = extract_sql(response.output_text)
    result = connection.sql(sql).execute()

    separator = "-" * 72
    print(f"Question:\n{question}\n")
    print(f"Generated SQL:\n{separator}\n{sql}\n{separator}")
    print("\nResult:")
    print(result.to_string(index=False))

    return sql, result

In [22]:
output = ask_database(question=question)

Question:
What was the total passenger count in 2024?

Generated SQL:
------------------------------------------------------------------------
SELECT SUM("Passenger Count") AS "total_passenger_count"
FROM "air_traffic"
WHERE "Year" = 2024;
------------------------------------------------------------------------

Result:
total_passenger_count
             52210939


## Checkpoint

We now have a minimal SQL AI agent that can receive a question, add the table schema as context, ask a model for DuckDB SQL, extract the query, and execute it through Ibis.

This deliberately small example sends model-generated SQL directly to the database. It is useful for understanding the end-to-end flow, but it is not a production safety pattern. Later tutorials will add richer context, a stronger output contract, SQL validation, bounded retries, and independently enforced database permissions.